## Optuna In Deckard: Studies, Trials, Samplers, Pruners, And Storage

This notebook mirrors the Hydra composition style used in the Hydra walkthrough and then focuses on Optuna concepts end-to-end.

What you will see:

- how Deckard config composition defines optimization directions and objectives,
- what studies and trials represent in Optuna,
- how samplers choose candidate parameters,
- how pruners stop weak trials early,
- how multi-objective optimization works,
- how trial data is persisted in a SQLite database.

In [1]:
import sqlite3
import tempfile
from pathlib import Path

import optuna
from hydra import compose, initialize_config_dir
from hydra.core.config_store import ConfigStore
from hydra.core.global_hydra import GlobalHydra

from deckard.layers.optimize import OptunaStudyCallback

PROJECT_ROOT = Path("../..").resolve()
CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"

def reset_hydra_state() -> None:
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_store = ConfigStore.instance()
    for key in list(config_store.repo.keys()):
        if key not in {"hydra", "_dummy_empty_config_.yaml"}:
            config_store.repo.pop(key, None)

reset_hydra_state()
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    cfg = compose(
        config_name="default",
        overrides=["score=classification"],
        return_hydra_config=True,
    )

print("Config dir:", CONFIG_DIR)
print("Directions from config:", list(cfg.directions))
print("Objectives (optimizers) from config:", list(cfg.optimizers))
print("Hydra sweeper sampler target:", cfg.hydra.sweeper.sampler._target_)
print("Hydra sweeper study template:", cfg.hydra.sweeper.study_name)

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config dir: /Users/c.meyers/Documents/deckard/examples/sklearn/config
Directions from config: ['maximize', 'maximize', 'maximize']
Objectives (optimizers) from config: ['accuracy', 'evasion_accuracy', 'attack_generation_time']
Hydra sweeper sampler target: optuna.samplers.RandomSampler
Hydra sweeper study template: adult_rf_class-labels_hsj


## Core Concepts: Study, Trial, Sampler, Pruner

- A study is the full optimization experiment.
- A trial is one candidate evaluation inside that study.
- A sampler proposes the next parameter values.
- A pruner can stop underperforming trials before they finish.

In the next cell, we run a small single-objective study with a TPE sampler and a median pruner.

In [2]:
sampler = optuna.samplers.TPESampler(seed=7)
pruner = optuna.pruners.MedianPruner(n_startup_trials=1, n_warmup_steps=1)

single_study = optuna.create_study(
    study_name="single_objective_demo",
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    storage=optuna.storages.InMemoryStorage(),
)

def single_objective(trial: optuna.Trial) -> float:
    x = trial.suggest_float("x", -2.0, 2.0)
    score = 1.0 - (x - 0.5) ** 2
    for step in range(2):
        intermediate = score - (0.15 * (1 - step))
        trial.report(intermediate, step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return score

single_study.optimize(single_objective, n_trials=6)

state_counts = {}
for trial in single_study.trials:
    state_counts[trial.state.name] = state_counts.get(trial.state.name, 0) + 1

print("Single-objective study:", single_study.study_name)
print(" Best value:", round(single_study.best_value, 4))
print(" Best params:", single_study.best_params)
print(" Trial states:", state_counts)

[I 2026-05-19 09:03:17,962] A new study created in memory with name: single_objective_demo


[I 2026-05-19 09:03:17,963] Trial 0 finished with value: -3.81700149295573 and parameters: {'x': -1.6947668425041713}. Best is trial 0 with value: -3.81700149295573.


[I 2026-05-19 09:03:17,964] Trial 1 finished with value: 0.6160026849738272 and parameters: {'x': 1.1196751689604585}. Best is trial 1 with value: 0.6160026849738272.


[I 2026-05-19 09:03:17,965] Trial 2 finished with value: 0.44294216141635123 and parameters: {'x': -0.246363074236426}. Best is trial 1 with value: 0.6160026849738272.


[I 2026-05-19 09:03:17,965] Trial 3 finished with value: 0.844873740075538 and parameters: {'x': 0.8938607113237649}. Best is trial 3 with value: 0.844873740075538.


[I 2026-05-19 09:03:17,966] Trial 4 pruned. 


[I 2026-05-19 09:03:17,966] Trial 5 finished with value: 0.8802725690232243 and parameters: {'x': 0.1539834816417347}. Best is trial 5 with value: 0.8802725690232243.


Single-objective study: single_objective_demo
 Best value: 0.8803
 Best params: {'x': 0.1539834816417347}
 Trial states: {'COMPLETE': 5, 'PRUNED': 1}


## Multi-Objective Optimization

Multi-objective studies optimize several goals at once. Instead of one best value, Optuna returns a Pareto front of non-dominated trials.

Deckard commonly uses multiple objectives (for example, performance and robustness metrics), so this pattern is central to sweep runs.

In [3]:
multi_study = optuna.create_study(
    study_name="multi_objective_demo",
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=11),
    storage=optuna.storages.InMemoryStorage(),
)

def multi_objective(trial: optuna.Trial) -> tuple[float, float]:
    x = trial.suggest_float("x", -2.0, 2.0)
    y = trial.suggest_float("y", -2.0, 2.0)
    objective_1 = -(x - 1.0) ** 2 - (y + 0.5) ** 2 + 2.0
    objective_2 = abs(x) + abs(y)
    return objective_1, objective_2

multi_study.optimize(multi_objective, n_trials=12)

pareto = multi_study.best_trials
print("Multi-objective study:", multi_study.study_name)
print(" Directions:", [d.name for d in multi_study.directions])
print(" Pareto trial count:", len(pareto))
print(" Example Pareto values:", pareto[0].values if pareto else None)

[I 2026-05-19 09:03:17,971] A new study created in memory with name: multi_objective_demo


[I 2026-05-19 09:03:17,971] Trial 0 finished with values: [-5.2158477012457, 3.201020278542425] and parameters: {'x': -1.2789212444929232, 'y': -1.9220990340495017}.


[I 2026-05-19 09:03:17,972] Trial 1 finished with values: [-1.2751578934981729, 1.046861610775213] and parameters: {'x': -0.14712589400662157, 'y': 0.8997357167685913}.


[I 2026-05-19 09:03:17,972] Trial 2 finished with values: [0.06464309701303872, 0.3774771889779607] and parameters: {'x': -0.3191855816490903, 'y': -0.0582916073288704}.


[I 2026-05-19 09:03:17,973] Trial 3 finished with values: [-6.8979120874774065, 1.999390312358079] and parameters: {'x': -1.9488767416375654, 'y': -0.050513570720513545}.


...

[I 2026-05-19 09:03:17,974] Trial 8 finished with values: [-7.582300760693403, 3.4511164737150044] and parameters: {'x': -1.9180655488350693, 'y': -1.533050924879935}.


[I 2026-05-19 09:03:17,975] Trial 9 finished with values: [-1.7626300005013165, 2.1028815268500907] and parameters: {'x': -0.7345307535498384, 'y': -1.3683507733002522}.


[I 2026-05-19 09:03:17,975] Trial 10 finished with values: [-1.1451788142898685, 2.3090197840985955] and parameters: {'x': 1.0359183526420215, 'y': 1.273101431456574}.


[I 2026-05-19 09:03:17,975] Trial 11 finished with values: [-0.6798060570269375, 1.3463068487133958] and parameters: {'x': -0.6215020361631303, 'y': -0.7248048125502655}.


Multi-objective study: multi_objective_demo
 Directions: ['MAXIMIZE', 'MINIMIZE']
 Pareto trial count: 2
 Example Pareto values: [0.06464309701303872, 0.3774771889779607]


## Database Storage: Persisting Studies And Trials

For reproducibility and analysis, Optuna can persist state in a database (for example SQLite, PostgreSQL, MySQL).

In the next cell, we use a temporary SQLite file, run a few trials, and inspect the tables directly.

In [4]:
tmp_db = Path(tempfile.gettempdir()) / "optuna_notebook_demo.db"
if tmp_db.exists():
    tmp_db.unlink()

storage_url = f"sqlite:///{tmp_db}"
db_study = optuna.create_study(
    study_name="sqlite_demo",
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(seed=3),
    storage=storage_url,
    load_if_exists=True,
)

def db_objective(trial: optuna.Trial) -> float:
    a = trial.suggest_float("a", 0.0, 1.0)
    b = trial.suggest_float("b", 0.0, 1.0)
    return 1.0 - ((a - 0.3) ** 2 + (b - 0.7) ** 2)

db_study.optimize(db_objective, n_trials=5)

with sqlite3.connect(tmp_db) as conn:
    tables = [row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    study_rows = conn.execute("SELECT COUNT(*) FROM studies").fetchone()[0]
    trial_rows = conn.execute("SELECT COUNT(*) FROM trials").fetchone()[0]

print("SQLite path:", tmp_db)
print("Table count:", len(tables))
print("Key tables present:", [name for name in ["studies", "trials", "trial_values", "trial_params"] if name in tables])
print("Study rows:", study_rows, "| Trial rows:", trial_rows)

[I 2026-05-19 09:03:18,440] A new study created in RDB with name: sqlite_demo


[I 2026-05-19 09:03:18,461] Trial 0 finished with value: 0.9370340250507776 and parameters: {'a': 0.5507979025745755, 'b': 0.7081478226181048}. Best is trial 0 with value: 0.9370340250507776.


[I 2026-05-19 09:03:18,472] Trial 1 finished with value: 0.9641310812705071 and parameters: {'a': 0.2909047389129443, 'b': 0.510827605197663}. Best is trial 1 with value: 0.9641310812705071.


[I 2026-05-19 09:03:18,483] Trial 2 finished with value: 0.6098829325668097 and parameters: {'a': 0.8929469543476547, 'b': 0.8962930889334381}. Best is trial 1 with value: 0.9641310812705071.


[I 2026-05-19 09:03:18,493] Trial 3 finished with value: 0.7267699349284658 and parameters: {'a': 0.12558531046383625, 'b': 0.20724287813818676}. Best is trial 1 with value: 0.9641310812705071.


[I 2026-05-19 09:03:18,503] Trial 4 finished with value: 0.8710519118164815 and parameters: {'a': 0.05146720330082988, 'b': 0.44080984365063647}. Best is trial 1 with value: 0.9641310812705071.


SQLite path: /var/folders/cd/frntyg595hj5gz_qzm3696hm0000gp/T/optuna_notebook_demo.db
Table count: 13
Key tables present: ['studies', 'trials', 'trial_values', 'trial_params']
Study rows: 1 | Trial rows: 5


## Deckard Callback Context

Deckard uses a callback during optimization runs to persist useful study metadata (including objective naming when configured).

This mirrors the Hydra defaults wiring shown in the Hydra notebook, where callback and sweeper settings are composed from config groups.

In [5]:
callback = OptunaStudyCallback(
    study_name="deckard_callback_demo",
    storage="sqlite:///optuna.db",
    directions=list(cfg.directions),
    optimizers=list(cfg.optimizers),
)

print("Callback class:", type(callback).__name__)
print("Configured directions:", list(cfg.directions))
print("Configured objectives:", list(cfg.optimizers))
print("Note: objective names are persisted when the callback is active in Hydra-driven runs.")

Callback class: OptunaStudyCallback
Configured directions: ['maximize', 'maximize', 'maximize']
Configured objectives: ['accuracy', 'evasion_accuracy', 'attack_generation_time']
Note: objective names are persisted when the callback is active in Hydra-driven runs.


In [6]:
import json
import pickle
from pathlib import Path

ARTIFACT_DIR = Path('build/notebook_artifacts/optuna')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# Save single-objective study
with open(ARTIFACT_DIR / 'single_study.pkl', 'wb') as f:
    pickle.dump(single_study, f)
# Save multi-objective study
with open(ARTIFACT_DIR / 'multi_study.pkl', 'wb') as f:
    pickle.dump(multi_study, f)
# Save database study summary
db_summary = {
    'sqlite_path': str(tmp_db),
    'table_count': len(tables),
    'key_tables': [name for name in ['studies', 'trials', 'trial_values', 'trial_params'] if name in tables],
    'study_rows': study_rows,
    'trial_rows': trial_rows,
} 
with open(ARTIFACT_DIR / 'db_summary.json', 'w', encoding='utf-8') as f:
    json.dump(db_summary, f, indent=2)
# Save best params from single and multi studies
with open(ARTIFACT_DIR / 'single_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(single_study.best_params, f, indent=2)
with open(ARTIFACT_DIR / 'multi_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(pareto[0].params if pareto else {}, f, indent=2)